# 📊 Notebook 01: 电力数据获取 — 山东 15 分钟级数据

## Data Ingestion: Shandong 15-Minute Power Data

## 学习目标 (Learning Objectives)
1. 理解为什么需要高时间分辨率电力数据
2. 学会用 DataLoader 加载山东 15 分钟级电力数据
3. 掌握 DataFrame 基本探索和可视化技巧
4. 了解山东电力市场的多维度数据列

## 背景知识 (Background)

### 电力数据的时间粒度 (Temporal Granularity)

电力数据按时间粒度分为:
- **年级 (Yearly)**: 年度发电量、用电量，适合宏观趋势分析
- **月级 (Monthly)**: 月度供需数据，适合季节性分析
- **日级 (Daily)**: 每日负荷曲线，可以做简单的时序预测
- **小时级 (Hourly)**: 小时级负荷数据，捕获日周期模式
- **15 分钟级 (15-Minute)**: 接近实时，捕获更细粒度的波动——这是我们的新起点！

### 山东电力数据 (Shandong Power Data)

我们使用的是**真实山东电力现货市场数据**，来源为用户提供的 CSV。

**数据概况 (Data Profile):**
- 时间跨度: 2024-01-01 ~ 2026-01-14 (约 745 天)
- 时间粒度: **15 分钟** (96 个数据点/天)
- 总行数: 约 71,520 行
- 列数: 21 列 (负荷、风电、光伏、核电、价格等)

**为什么选山东？**
1. 🔢 **高分辨率** — 96 点/天，比小时数据精细 4 倍
2. ⚡ **多维度** — 负荷 + 风光核 + 价格，完整的电力市场画像
3. 💰 **含价格** — 日前/实时电价，可直接模拟交易
4. 🌍 **风光大省** — 山东是中国光伏/风电装机大省，新能源占比高

> 💡 **思考 (Think)**:
> 15 分钟数据 vs 小时数据，对负荷预测有什么额外价值？
> 电力现货市场为什么需要 15 分钟结算？

In [ ]:
# ── 导入依赖 (Imports) ──────────────────────────
from ellectric.pipeline.data_loader import create_loader
from ellectric.pipeline.cleaner import clean_data, validate_schema
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook'  # 在 Jupyter 中渲染交互图表

---
## 步骤 1: 加载山东 15 分钟数据 (Load Shandong 15-Min Data)

调用 `create_loader('shandong')` 创建山东数据加载器，
然后 `.load_data()` 加载指定时间范围的数据。

### 数据加载器的工作原理 (How DataLoader Works)
```
create_loader('shandong')
  → ShandongDataLoader.__init__()
  → load_data(start, end)
    → pd.read_csv() 读取 CSV (UTF-8-sig)
    → 列重命名: 日期+时刻→timestamp, 直调负荷(实际)→load_mw, ...
    → 时间戳构造: "2024-06-01 00:00" → datetime64[ns, UTC]
    → 类型转换: 节假日标记 "是"/"否" → 1/0
    → 返回标准化 DataFrame
```

### 选择时间范围 (Time Range)

本教程使用 **2024 年夏季 (6月-9月)** 的数据:
- 6月: 夏季初期，空调负荷开始上升
- 7-8月: 盛夏高峰，制冷负荷最大
- 9月: 夏季尾部，负荷逐渐回落
- 共约 4 个月 × 30 天 × 96 点 = 约 11,520 行

In [ ]:
# 创建山东加载器，拉取 2024 夏季数据
loader = create_loader('shandong')
df_raw = loader.load_data('2024-06-01', '2024-09-30')

# 查看数据概况
print(f"数据形状: {df_raw.shape[0]} 行 × {df_raw.shape[1]} 列")
print(f"时间范围: {df_raw['timestamp'].min()} ~ {df_raw['timestamp'].max()}")
print(f"时间粒度: 15 分钟 (96 点/天)")
print(f"预期总行数: 122 天 × 96 点 = 11,712 (实际 {df_raw.shape[0]})")
df_raw.head(10)

### 📖 数据列说明 (Column Reference)

山东数据比 OWID 年级数据丰富得多。以下是核心列:

| 列名 | 中文含义 | 单位 |
|------|---------|------|
| `timestamp` | 时间戳 (UTC) | datetime64 |
| `load_mw` | 直调负荷 (实际) | MW |
| `rt_price` | 实时电价 | 元/MWh |
| `da_price` | 日前电价 (~75% null) | 元/MWh |
| `wind_actual_mw` | 风电实际出力 | MW |
| `solar_actual_mw` | 光伏实际出力 | MW |
| `nuclear_actual_mw` | 核电实际出力 | MW |
| `tie_line_actual_mw` | 联络线受电 | MW |
| `pumped_storage_mw` | 抽水蓄能 | MW |
| `is_holiday` | 是否节假日 | 0/1 |
| `is_weekend` | 是否周末 | 0/1 |
| `province` | 省份 | "shandong" |
| `source` | 数据来源 | "user-provided-csv" |
| `granularity` | 时间粒度 | "15min" |

**15min 数据 vs 1h 数据:**

每小时 1 个数据点 → 每天 24 个点 → 粗粒度，丢失日内波动细节
每 15 分钟 1 个点 → 每天 96 个点 → 细粒度，捕获更精确的爬坡/负荷变化

对于机器学习模型来说，**更多数据 = 更多学习信号**。
96 点/天的数据量是小时级数据的 4 倍，意味着模型可以看到更细微的负荷波动模式。

In [ ]:
# ── 数据统计概览 (Statistical Overview) ──────────
print("═══ 核心列统计 ═══")
core_cols = ['load_mw', 'rt_price', 'wind_actual_mw', 'solar_actual_mw']
print(df_raw[core_cols].describe())

print(f"\n═══ 缺失值概览 ═══")
missing = df_raw.isna().sum()
missing_pct = (missing / len(df_raw) * 100)
missing_df = pd.DataFrame({'缺失数': missing, '百分比': missing_pct})
print(missing_df[missing_df['缺失数'] > 0].to_string())

print(f"\n═══ 数据源元信息 ═══")
print(f"省份: {df_raw['province'].iloc[0]}")
print(f"粒度: {df_raw['granularity'].iloc[0]}")
print(f"来源: {df_raw['source'].iloc[0]}")

---
## 步骤 2: 数据可视化 — 15 分钟级探索 (Visual Exploration)

与年级数据不同，15 分钟数据可以让我们看到**一周内的日内波动**。
这里选取第一周数据（7天 × 96点 = 672点），观察:
1. 负荷的日周期模式（早晚高峰、凌晨低谷）
2. 风电/光伏的日内变化
3. 实时电价与负荷的关系

In [ ]:
# ── 第一周负荷曲线 (7 天 × 96 点 = 672 点) ─────
first_week = df_raw.iloc[:672].copy()

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=('直调负荷 + 风电/光伏出力 (第一周)', '实时电价'),
    row_heights=[0.55, 0.45],
    vertical_spacing=0.1
)

# 子图1: 负荷 + 风光
fig.add_trace(
    go.Scatter(x=first_week['timestamp'], y=first_week['load_mw'],
               name='负荷', mode='lines',
               line=dict(color='#1f77b4', width=1.5)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=first_week['timestamp'], y=first_week['wind_actual_mw'],
               name='风电', mode='lines',
               line=dict(color='#00CED1', width=1)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=first_week['timestamp'], y=first_week['solar_actual_mw'],
               name='光伏', mode='lines',
               line=dict(color='#FFD700', width=1)),
    row=1, col=1
)

# 子图2: 实时电价
fig.add_trace(
    go.Scatter(x=first_week['timestamp'], y=first_week['rt_price'],
               name='实时电价', mode='lines',
               line=dict(color='#d62728', width=1)),
    row=2, col=1
)

fig.update_layout(
    title='山东电力数据 — 第一周 (15min 粒度)',
    hovermode='x unified',
    height=600,
    showlegend=True
)
fig.update_yaxes(title_text='MW', row=1, col=1)
fig.update_yaxes(title_text='元/MWh', row=2, col=1)
fig.update_xaxes(title_text='时间', row=2, col=1)
fig.show()

### 📖 从图表中你能看到什么？(What Can You See?)

1. **日周期模式 (Diurnal Cycle)**
   负荷每天有两个高峰——上午 10-12 点和下午 17-19 点，凌晨 3-5 点为低谷。
   15 分钟粒度让这些波动清晰可见，而不是被平均掉。

2. **光伏的"钟形曲线" (Solar Bell Curve)**
   光伏出力每天从日出到日落呈现完美的钟形曲线——晴天的典型特征。

3. **风电的间歇性 (Wind Intermittency)**
   风电不像光伏那样规律——有时白天多有时晚上多，取决于天气系统。

4. **电价与负荷的关系 (Price-Load Relationship)**
   实时电价往往在负荷高峰时段更高——供需关系的基本经济学原理。

> 💡 **思考 (Think)**:
> 如果你是电力交易员，15 分钟数据和小时数据在决策上有什么不同？
> 光伏中午出力最大，电价中午反而可能更低（供给过剩）——这叫"鸭子曲线"效应。

---
## 步骤 3: 数据清洗预览 (Cleaning Preview)

在进入正式的清洗流程前 (Notebook 02)，先用 `clean_data()` 快速验证:

In [ ]:
df = clean_data(df_raw)

# 验证清洗结果
result = validate_schema(df)
print(f"Schema 验证: {'✅ 通过' if result['valid'] else '❌ 失败'}")
if result['issues']:
    for issue in result['issues']:
        print(f"  ! {issue}")

print(f"\n清洗后数据概况:")
print(f"  行数: {result['summary']['rows']}")
print(f"  时间范围: {result['summary']['start']} ~ {result['summary']['end']}")
print(f"  负荷范围: {result['summary']['load_min']:.0f} ~ {result['summary']['load_max']:.0f} MW")
print(f"  日均负荷: {result['summary']['load_min']:.0f} ~ {result['summary']['load_max']:.0f} MW (日内波动)")
df.head()

### 平均日负荷曲线 (Average Daily Profile)

将 4 个月的数据按 15 分钟时段（00:00, 00:15, ..., 23:45）分组平均，
得到一条"典型日"的负荷曲线——这是电力系统的经典分析工具。

In [ ]:
# ── 平均日负荷曲线 ──────────────────────────────
# 提取每个 15 分钟时段的小时+分钟标签
df_clean = df.copy()
df_clean['time_of_day'] = df_clean['timestamp'].dt.strftime('%H:%M')

# 按一天中的时刻分组，计算平均负荷
daily_profile = df_clean.groupby('time_of_day')['load_mw'].mean()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=daily_profile.index,
    y=daily_profile.values,
    mode='lines',
    name='平均负荷',
    line=dict(color='#1f77b4', width=2),
    fill='tozeroy',
    fillcolor='rgba(31, 119, 180, 0.15)'
))

# 标注早晚高峰
fig.add_vline(x='10:00', line_dash='dash', line_color='red',
              annotation_text='早高峰 ≈10:00')
fig.add_vline(x='18:00', line_dash='dash', line_color='red',
              annotation_text='晚高峰 ≈18:00')

fig.update_layout(
    title='山东平均日负荷曲线 (2024 夏季, 15min 粒度)',
    xaxis_title='时刻',
    yaxis_title='平均负荷 (MW)',
    hovermode='x',
    height=450
)
fig.update_xaxes(tickangle=45, dtick=16)  # 每 4 小时一个刻度 (16×15min)
fig.show()

---
## 📝 学习笔记 (Key Takeaways)

- 我们成功加载了山东 2024 夏季 15 分钟级电力数据（约 11,520 行）
- 数据包含负荷、风光核出力、实时/日前电价等 **14+ 数值列**
- 15 分钟粒度让你看到小时内波动——这是小时数据无法提供的
- 日负荷明显呈现双峰模式（早 10 点 + 晚 18 点），这是工业+居民混合型负荷特征
- `DataLoader` 抽象层让你无缝在不同数据源间切换

**下一步: Notebook 02 → 深入数据清洗**

### 思考题 (Reflection Questions)

1. 这个数据集每天 96 个点。如果把每小时的前 15 分钟和后 15 分钟的负荷差作为"爬坡速度"，你能看到什么模式？
2. 光伏出力的钟形曲线和负荷的双峰曲线，在时间上是错开的还是重叠的？这对电力调度意味着什么？
3. 为什么 `da_price` (日前电价) 有 ~75% 缺失？实际交易中日前电价和实时电价有什么区别？